# Training Pipeline — Baseline Variant

Same architecture as the LSH variant, trained instead on the full (non-distilled) training set — the assertiveness-focused baseline reported in the thesis. Outputs `.joblib` model packages to `../models/baseline/`.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import pyreadr
import shap
import optuna
import xgboost as xgb
import time
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")

# ==============================================================================
# NOTE: paths below are relative to this notebook living in /notebooks.
# Place the TEP dataset files in /data and trained model artifacts in /models
# (see the repository README for download links and folder layout).
# 0. CONFIGURAÇÕES GERAIS
# ==============================================================================
PASTA_MODELOS = os.path.join("..", "models", "baseline")
CAMINHO_NORMAL = os.path.join("..", "data", "tep_fault_free_cache.csv")
CAMINHO_FALHAS = os.path.join("..", "data", "TEP_Faulty_Training.RData")
FALHAS_PARA_IGNORAR = [3, 9, 15]

USAR_CUDA = True  # mude para False se não tiver CUDA disponível

if not os.path.exists(PASTA_MODELOS):
    os.makedirs(PASTA_MODELOS)

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==============================================================================
# 1. CARREGAMENTO GLOBAL
# ==============================================================================
print("⏳ Carregando bases de referência...")
df_normal = pd.read_csv(CAMINHO_NORMAL)
res_r = pyreadr.read_r(CAMINHO_FALHAS)
df_todas_falhas = res_r[list(res_r.keys())[0]]

# ==============================================================================
# FUNÇÕES AUXILIARES
# ==============================================================================
def criar_modelo_xgb(params=None):
    base_params = {
        "tree_method": "hist",
        "random_state": 42,
        "eval_metric": "logloss"
    }
    if USAR_CUDA:
        base_params["device"] = "cuda"
    if params:
        base_params.update(params)
    return xgb.XGBClassifier(**base_params)

def preparar_base(df, prefixo, target_value):
    df = df.copy()
    df["target"] = target_value

    # group_id único por origem + faultNumber + simulationRun
    if "faultNumber" in df.columns:
        df["group_id"] = (
            prefixo + "_" +
            df["faultNumber"].astype(str) + "_" +
            df["simulationRun"].astype(str)
        )
    else:
        df["group_id"] = prefixo + "_" + df["simulationRun"].astype(str)

    return df

def split_por_grupo(df, train_frac=0.7, seed=42, group_col="group_id"):
    grupos = df[group_col].drop_duplicates().to_numpy()
    rng = np.random.RandomState(seed)
    grupos = rng.permutation(grupos)

    corte = int(len(grupos) * train_frac)
    grupos_treino = set(grupos[:corte])

    df_train = df[df[group_col].isin(grupos_treino)].copy()
    df_test = df[~df[group_col].isin(grupos_treino)].copy()

    return df_train, df_test

# Tenta usar split estratificado por grupo; se não existir, cai no GroupKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    TEM_STRATIFIED_GROUP = True
except ImportError:
    from sklearn.model_selection import GroupKFold
    TEM_STRATIFIED_GROUP = False

# ==============================================================================
# 2. LOOP COMPLETO: ESPECIALISTAS DE 1 A 20
# ==============================================================================

# --- ADICIONAR ANTES DO LOOP ---
print("⏳ Preparando Split Global (Garantindo mesma base de teste do LSH)...")
df_normal['faultNumber'] = 0
df_todas_falhas = df_todas_falhas[df_todas_falhas['sample'] > 20].copy()

df_normal["group_id"] = "normal_f0_run" + df_normal["simulationRun"].astype(str)
df_todas_falhas["group_id"] = "falha_f" + df_todas_falhas["faultNumber"].astype(str) + "_run" + df_todas_falhas["simulationRun"].astype(str)

df_global = pd.concat([df_todas_falhas, df_normal], ignore_index=True)
df_train_global, df_test_global = split_por_grupo(df_global, train_frac=0.7, seed=42)

estatisticas_gerais = [] # Preparando a lista para o relatório final
tempo_inicio_global = time.time() # Não esqueça de importar o 'time' lá em cima!
for f_id in range(1, 21):
    if f_id in FALHAS_PARA_IGNORAR:
        print(f"⏩ Pulando Falha {f_id} (Classe de Baixa Separabilidade).")
        continue

    print(f"\n=======================================================")
    print(f"🚀 TREINANDO ESPECIALISTA DE ELITE: FALHA {f_id}")
    print(f"=======================================================")

    try:
        tempo_inicio_esp = time.time() # Começa a cronometrar o especialista
        
        # ======================================================================
        # 2.1 CONSTRUÇÃO DO TREINO (Puxando do Split Global)
        # ======================================================================
        # Pega as amostras da falha alvo no Cofre de Treino e marca como 1
        df_alvo_tr = df_train_global[df_train_global["faultNumber"] == f_id].copy()
        df_alvo_tr["target"] = 1

        # Pega todo o resto (Normal e outras falhas) no Cofre de Treino e marca como 0
        df_outras_tr = df_train_global[df_train_global["faultNumber"] != f_id].copy()
        df_outras_tr["target"] = 0

        # Cria o df_train_raw para o seu código de balanceamento usar
        df_train_raw = pd.concat([df_alvo_tr, df_outras_tr], ignore_index=True)

        # ======================================================================
        # 2.2 CONSTRUÇÃO DO TESTE (Puxando do Split Global)
        # ======================================================================
        df_alvo_ts = df_test_global[df_test_global["faultNumber"] == f_id].copy()
        df_alvo_ts["target"] = 1

        df_outras_ts = df_test_global[df_test_global["faultNumber"] != f_id].copy()
        df_outras_ts["target"] = 0

        # Cria o df_test_raw para o seu código de teste usar
        df_test_raw = pd.concat([df_alvo_ts, df_outras_ts], ignore_index=True)


        # ======================================================================
        # 2.3 BALANCEAMENTO SEGURO
        # ======================================================================

        df_train_pos = df_train_raw[df_train_raw["target"] == 1].copy()
        df_train_neg_pool = df_train_raw[df_train_raw["target"] == 0].copy()

        df_test_pos = df_test_raw[df_test_raw["target"] == 1].copy()
        df_test_neg_pool = df_test_raw[df_test_raw["target"] == 0].copy()

        if len(df_train_pos) == 0 or len(df_train_neg_pool) == 0:
            print(f"⚠️ Falha {f_id}: treino sem classes suficientes. Pulando.")
            continue

        if len(df_test_pos) == 0 or len(df_test_neg_pool) == 0:
            print(f"⚠️ Falha {f_id}: teste sem classes suficientes. Pulando.")
            continue

        n_train = min(len(df_train_pos), len(df_train_neg_pool))
        n_test = min(len(df_test_pos), len(df_test_neg_pool))

        df_train_full = pd.concat([
            df_train_pos.sample(n=n_train, random_state=42),
            df_train_neg_pool.sample(n=n_train, random_state=f_id) # <--- MUDE AQUI PARA f_id
        ], ignore_index=True).sort_values(["group_id", "sample"]).reset_index(drop=True)

        df_test_full = pd.concat([
            df_test_pos.sample(n=n_test, random_state=42),
            df_test_neg_pool.sample(n=n_test, random_state=42)
        ], ignore_index=True).sort_values(["group_id", "sample"]).reset_index(drop=True)

        # ======================================================================
        # MATRIZES FINAIS
        # ======================================================================
        drop_cols = ["faultNumber", "simulationRun", "sample", "target", "group_id"]

        X_train_base = df_train_full.drop(columns=drop_cols, errors="ignore")
        y_train_base = df_train_full["target"].astype(int)
        groups_train = df_train_full["group_id"]

        X_test_base = df_test_full.drop(columns=drop_cols, errors="ignore")
        y_test_base = df_test_full["target"].astype(int)

        n_groups_train = groups_train.nunique()
        if n_groups_train < 2:
            print(f"⚠️ Falha {f_id}: poucos grupos no treino para CV. Pulando.")
            continue

        # ======================================================================
        # FASE A: BASELINE
        # ======================================================================
        modelo_base = criar_modelo_xgb()
        modelo_base.fit(X_train_base, y_train_base)

        y_pred_base = modelo_base.predict(X_test_base)
        acc_base = accuracy_score(y_test_base, y_pred_base)

        # ======================================================================
        # FASE B: SHAP
        # ======================================================================
        print("🔍 Executando SHAP...")
        explainer = shap.TreeExplainer(modelo_base)

        X_shap_sample = X_train_base.sample(
            n=min(2000, len(X_train_base)),
            random_state=42
        )

        shap_values = explainer.shap_values(X_shap_sample)

        # Compatibilidade com versões diferentes do SHAP
        if isinstance(shap_values, list):
            shap_values = shap_values[-1]

        shap_values = np.asarray(shap_values)
        if shap_values.ndim == 3:
            shap_values = shap_values[:, :, -1]

        top_indices = (
            pd.Series(np.abs(shap_values).mean(axis=0), index=X_train_base.columns)
            .nlargest(min(10, X_train_base.shape[1]))
            .index.tolist()
        )

        print(f"✅ Top 10 Variáveis Selecionadas: {', '.join(top_indices)}")

        # ======================================================================
        # FASE C: OTIMIZAÇÃO E TREINO FINAL
        # ======================================================================
        X_train_top = X_train_base[top_indices]
        X_test_final = X_test_base[top_indices]

        def objective(trial):
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 300),
                "max_depth": trial.suggest_int("max_depth", 4, 7),
                "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.2)
            }

            model = criar_modelo_xgb(params)

            if TEM_STRATIFIED_GROUP:
                n_splits_cv = min(3, n_groups_train)
                cv = StratifiedGroupKFold(
                    n_splits=n_splits_cv,
                    shuffle=True,
                    random_state=42
                )
            else:
                from sklearn.model_selection import GroupKFold
                n_splits_cv = min(3, n_groups_train)
                cv = GroupKFold(n_splits=n_splits_cv)

            scores = cross_val_score(
                model,
                X_train_top,
                y_train_base,
                cv=cv,
                groups=groups_train,
                scoring="accuracy"
            )
            return scores.mean()

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=10)

        best_model = criar_modelo_xgb(study.best_params)
        best_model.fit(X_train_top, y_train_base)

        y_pred_final = best_model.predict(X_test_final)

        acc_final = accuracy_score(y_test_base, y_pred_final)
        prec_final = precision_score(y_test_base, y_pred_final, zero_division=0)
        rec_final = recall_score(y_test_base, y_pred_final, zero_division=0)
        f1_final = f1_score(y_test_base, y_pred_final, zero_division=0)


        # ======================================================================
        # FASE D: RELATÓRIO E SALVAMENTO
        # ======================================================================
        tempo_fim_esp = time.time()
        tempo_gasto_esp = tempo_fim_esp - tempo_inicio_esp
        
        estatisticas_gerais.append({
            'falha': f_id,
            'amostras_treinadas': len(X_train_base),
            'tempo_segundos': tempo_gasto_esp,
            'acuracia': acc_final,
            'precisao': prec_final,
            'recall': rec_final,
            'f1': f1_final
        })

        print(f"\n📊 RELATÓRIO DE DESEMPENHO (FALHA {f_id}):")
        print(f"   ⏱️ Tempo de Treino do Especialista: {tempo_gasto_esp:.2f} segundos") # <--- Opcional: Adicione essa linha no seu print
        print(f"   📉 Acurácia Baseline (Todas as {X_train_base.shape[1]} variáveis): {acc_base*100:.2f}%")
        print(f"   📈 Acurácia Otimizada (Apenas as Top 10): {acc_final*100:.2f}%")
        print(f"   🎯 Precisão da Falha : {prec_final*100:.2f}%")
        print(f"   🔎 Recall da Falha   : {rec_final*100:.2f}%")
        print(f"   ⚖️ F1-Score          : {f1_final*100:.2f}%")

        pacote = {
            "falha_id": f_id,
            "features": top_indices,
            "modelo": best_model,
            "acuracia": acc_final,
            "precisao": prec_final,
            "recall": rec_final,
            "f1": f1_final
        }

        joblib.dump(pacote, os.path.join(PASTA_MODELOS, f"especialista_f{f_id}.joblib"))
        print(f"💾 Modelo salvo com sucesso!\n")

    except Exception as e:
        print(f"❌ Erro na Falha {f_id}: {e}")
        continue

# ==============================================================================
# 3. RELATÓRIO FINAL GLOBAL PARA O TCC
# ==============================================================================
tempo_fim_global = time.time()
tempo_total_minutos = (tempo_fim_global - tempo_inicio_global) / 60

media_tempo = np.mean([e['tempo_segundos'] for e in estatisticas_gerais])
media_f1 = np.mean([e['f1'] for e in estatisticas_gerais]) * 100
media_amostras = np.mean([e['amostras_treinadas'] for e in estatisticas_gerais])

print("\n" + "="*60)
print("🐢 RESUMO DO PIPELINE BASELINE (SEM LSH)")
print("="*60)
print(f"⏱️ Tempo Total de Execução: {tempo_total_minutos:.2f} minutos")
print(f"⏱️ Tempo Médio por Especialista : {media_tempo:.2f} segundos")
print(f"📦 Média de Amostras de Treino : {media_amostras:.0f} linhas")
print(f"⚖️ Média de F1-Score Global    : {media_f1:.2f}%")
print("="*60)